# N-Body CUDA / Barnes-Hut / CUDA-OpenGL Render

Run cells top to bottom. After the first run, re-run just the **pull + build** cell whenever the code changes, then re-run whatever test cell you need — no need to re-clone.

In [ ]:
!nvidia-smi

In [ ]:
# Headless EGL/OpenGL dev headers + libs, needed only for the renderer
# (nbody_cuda, the benchmark/correctness CLI, doesn't need any of this).
!apt-get update -qq
!apt-get install -y -qq libegl1-mesa-dev libgl1-mesa-dev libglvnd-dev libglvnd0 mesa-utils 2>&1 | tail -20

In [ ]:
import os
REPO_URL = "https://github.com/Pratik740/nbody-simulation.git"
BRANCH = "cuda-barnes-hut"
REPO_DIR = "/content/nbody-simulation"

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git reset --hard origin/{BRANCH}
%cd {REPO_DIR}/cuda
!bash build.sh

## Correctness check (small n)
Compares CPU reference vs GPU brute-force vs GPU Barnes-Hut on the same initial conditions. Brute-force should match the CPU reference to float-rounding precision; Barnes-Hut should be close but not identical (that gap is the theta approximation).

In [ ]:
!./build/nbody_cuda --mode=compare --scenario=orbit --n=3 --steps=500 --dt=0.01 --theta=0.5 --print-final

In [ ]:
!./build/nbody_cuda --mode=compare --scenario=random --n=500 --steps=100 --dt=0.005 --theta=0.5

## Benchmark: brute-force O(n^2) vs Barnes-Hut O(n log n)
Watch the ms/step column grow much faster for brute-force as n increases.

In [ ]:
for n in [1000, 5000, 20000, 100000]:
    print(f"--- n={n} ---")
    !./build/nbody_cuda --mode=bruteforce --scenario=random --n={n} --steps=20 --dt=0.005
    !./build/nbody_cuda --mode=barnes_hut --scenario=random --n={n} --steps=20 --dt=0.005 --theta=0.5

## CUDA-OpenGL interop render (zero-copy VBO)
Headless EGL context, physics runs on the GPU, positions are written straight into a mapped OpenGL VBO each frame (no CPU round trip), rendered to an off-screen framebuffer, and a handful of frames are dumped as PPM so we can look at them here. Watch stderr for `EGL ... initialized` and any `eglGetProcAddress failed` / `Framebuffer incomplete` lines — those pinpoint exactly where a headless-GL setup issue is, if there is one.

In [ ]:
!rm -rf frames && mkdir -p frames
!./build/nbody_render --n=20000 --frames=120 --mode=barnes_hut --theta=0.5 --dump-every=30 --dump-dir=frames

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

frames = sorted(glob.glob("frames/*.ppm"))
print(f"{len(frames)} frame(s) dumped: {frames}")
fig, axes = plt.subplots(1, len(frames), figsize=(6 * len(frames), 6))
if len(frames) == 1:
    axes = [axes]
for ax, path in zip(axes, frames):
    ax.imshow(Image.open(path))
    ax.set_title(path)
    ax.axis("off")
plt.show()

## Real-time check at scale
The renderer prints avg ms/frame and fps at the end. Try increasing `--n` (e.g. 200000, 1000000) to see where it stops being real-time (commonly accepted as 30+ fps) on this GPU -- that's the actual, measured answer to "can this do 1M+ bodies in real time on a T4", not an assumed one.

In [ ]:
for n in [100000, 500000, 1000000]:
    print(f"--- n={n} ---")
    !./build/nbody_render --n={n} --frames=120 --mode=barnes_hut --theta=0.7 --dump-every=0